In [ ]:
# ==============================================================================
# Bangla Sign Language (BDSL) Classification Pipeline using MobileNetV2
# ==============================================================================
# Description: This script loads the bdsl49_updated_2026 dataset, performs 
# 3-fold stratified cross-validation, trains a final robust model on the 
# combined train+validation split, and evaluates it on the test set. 
# Artifacts, metrics, and confusion matrices are saved automatically.
# ==============================================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==============================================================================
# 1. DIRECTORY CONFIGURATION (LOCAL PATHS)
# ==============================================================================
# Assumes 'bdsl49_updated_2026' is in the same directory as this script/notebook.
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "bdsl49_updated_2026")

train_dir = os.path.join(DATA_DIR, "train")
val_dir   = os.path.join(DATA_DIR, "val")
test_dir  = os.path.join(DATA_DIR, "test")

# Directory to save model checkpoints, evaluation metrics, and visualizations
OUT_DIR = os.path.join(BASE_DIR, "bdsl49_results")
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE = (224, 224)  # Standard input resolution for MobileNetV2
BATCH_SIZE = 32

# ==============================================================================
# 2. CLASS DISCOVERY & METADATA EXPORT
# ==============================================================================
# Automatically discover sign language classes from directory structure
classes = sorted(os.listdir(train_dir))
num_classes = len(classes)
print(f"Discovered {num_classes} sign language classes.")

# Save class labels list for downstream applications (e.g., Streamlit inference UI)
with open(os.path.join(OUT_DIR, "classes.json"), "w") as f:
    json.dump(classes, f, indent=2)

# Save explicit class-to-index mappings
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {str(i): c for i, c in enumerate(classes)}  # JSON keys require strings

class_mapping = {
    "class_to_idx": class_to_idx,
    "idx_to_class": idx_to_class
}
with open(os.path.join(OUT_DIR, "class_indices.json"), "w") as f:
    json.dump(class_mapping, f, indent=2)

# Save pipeline configuration metadata
pipeline_config = {
    "img_size": list(IMG_SIZE),
    "num_classes": num_classes,
    "preprocessing": "mobilenet_v2.preprocess_input"
}
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(pipeline_config, f, indent=2)

# ==============================================================================
# 3. DATA LOADING & DATAFRAME PREPARATION
# ==============================================================================
def load_dataset_paths(folder_path):
    """Recursively collects file paths and corresponding labels from folder directories."""
    data = []
    for class_name in classes:
        class_path = os.path.join(folder_path, class_name)
        if os.path.isdir(class_path):
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                if os.path.isfile(img_path):
                    data.append([img_path, class_name])
    return pd.DataFrame(data, columns=["path", "label"])

# Merge train and validation splits to pool data for cross-validation
print("Loading train and validation image paths...")
df = pd.concat([load_dataset_paths(train_dir), load_dataset_paths(val_dir)], ignore_index=True)

# Encode text labels into integer IDs
label_map = {c: i for i, c in enumerate(classes)}
df["label_id"] = df["label"].map(label_map)

X = df["path"].values
y = df["label_id"].values

# ==============================================================================
# 4. CUSTOM DATA GENERATOR WITH ON-THE-FLY AUGMENTATION
# ==============================================================================
def data_generator(files, labels, batch_size, augment=False):
    """Custom generator that yields batches of preprocessed images and labels."""
    while True:
        indices = np.random.permutation(len(files))
        for i in range(0, len(files), batch_size):
            batch_idx = indices[i:i + batch_size]
            X_batch, y_batch = [], []

            for j in batch_idx:
                try:
                    img = tf.keras.utils.load_img(files[j], target_size=IMG_SIZE)
                    img = tf.keras.utils.img_to_array(img)

                    # Apply lightweight augmentations only during training
                    if augment:
                        img = tf.image.random_flip_left_right(img)
                        img = tf.image.random_brightness(img, max_delta=0.1)

                    # Normalize pixel values using MobileNetV2 preprocessing specs
                    img = preprocess_input(img)

                    X_batch.append(img)
                    y_batch.append(labels[j])
                except Exception as e:
                    # Skip corrupt or unreadable image files gracefully
                    continue

            if X_batch:
                yield np.array(X_batch), np.array(y_batch)

# ==============================================================================
# 5. MODEL ARCHITECTURE BUILDER (TRANSFER LEARNING)
# ==============================================================================
def build_mobilenet_model():
    """Constructs MobileNetV2 with custom classification head and partial layer freezing."""
    base_model = MobileNetV2(
        include_top=False, 
        weights='imagenet',
        input_shape=(224, 224, 3), 
        pooling='avg'
    )

    # Freeze the initial 50 layers to retain low-level feature extractors
    for layer in base_model.layers[:50]:
        layer.trainable = False

    inputs = Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ==============================================================================
# 6. 3-FOLD STRATIFIED CROSS-VALIDATION
# ==============================================================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_metrics = []

print("\nStarting 3-Fold Stratified Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1}/3 ---")

    model = build_mobilenet_model()

    # Save architecture summary for documentation
    with open(os.path.join(OUT_DIR, f"model_summary_fold{fold + 1}.txt"), "w") as f:
        model.summary(print_fn=lambda line: f.write(line + "\n"))

    train_gen = data_generator(X[train_idx], y[train_idx], BATCH_SIZE, augment=True)
    val_gen   = data_generator(X[val_idx], y[val_idx], BATCH_SIZE, augment=False)

    steps_train = len(train_idx) // BATCH_SIZE
    steps_val   = len(val_idx) // BATCH_SIZE

    # Callbacks for checkpointing best weights and preventing overfitting
    ckpt_callback = ModelCheckpoint(
        os.path.join(OUT_DIR, f"best_fold_{fold + 1}.keras"),
        save_best_only=True,
        monitor="val_accuracy",
        verbose=1
    )
    early_stop = EarlyStopping(
        monitor="val_accuracy", 
        patience=3, 
        restore_best_weights=True
    )

    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=20,
        steps_per_epoch=steps_train,
        validation_steps=steps_val,
        callbacks=[ckpt_callback, early_stop],
        verbose=1
    )

    # Evaluate validation fold performance
    preds = model.predict(val_gen, steps=steps_val)
    y_pred = np.argmax(preds, axis=1)
    y_true = y[val_idx][:len(y_pred)]

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    fold_metrics.append([acc, prec, rec, f1])

# Save aggregated Cross-Validation report
fold_metrics = np.array(fold_metrics)
cv_report = {
    "accuracy_mean": float(np.mean(fold_metrics[:, 0])),
    "accuracy_std": float(np.std(fold_metrics[:, 0])),
    "precision_mean": float(np.mean(fold_metrics[:, 1])),
    "recall_mean": float(np.mean(fold_metrics[:, 2])),
    "f1_mean": float(np.mean(fold_metrics[:, 3]))
}

with open(os.path.join(OUT_DIR, "cv_results.json"), "w") as f:
    json.dump(cv_report, f, indent=2)

# ==============================================================================
# 7. FINAL MODEL TRAINING (FULL DATASET)
# ==============================================================================
print("\nTraining Final Model on the complete dataset...")

final_model = build_mobilenet_model()
final_gen = data_generator(X, y, BATCH_SIZE, augment=True)
total_steps = len(X) // BATCH_SIZE

final_model.fit(
    final_gen,
    epochs=20,
    steps_per_epoch=total_steps,
    verbose=1
)

final_model.save(os.path.join(OUT_DIR, "final_model.keras"))
with open(os.path.join(OUT_DIR, "final_model_summary.txt"), "w") as f:
    final_model.summary(print_fn=lambda line: f.write(line + "\n"))

# ==============================================================================
# 8. INDEPENDENT TEST SET EVALUATION & VISUALIZATION
# ==============================================================================
print("\nEvaluating model on the unseen test set...")

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
)
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False,
    classes=classes
)

test_preds = final_model.predict(test_gen)
y_pred_test = np.argmax(test_preds, axis=1)
y_true_test = test_gen.classes

# Generate and save Confusion Matrix heatmap
cm = confusion_matrix(y_true_test, y_pred_test)
plt.figure(figsize=(18, 18))
sns.heatmap(cm, cmap="Blues", xticklabels=classes, yticklabels=classes, cbar=True)
plt.title("Final Model Confusion Matrix - Bangla Sign Language", fontsize=14)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=300, bbox_inches='tight')
plt.close()

# Calculate final test metrics
test_metrics = {
    "accuracy": float(accuracy_score(y_true_test, y_pred_test)),
    "precision": float(precision_score(y_true_test, y_pred_test, average='macro', zero_division=0)),
    "recall": float(recall_score(y_true_test, y_pred_test, average='macro', zero_division=0)),
    "f1": float(f1_score(y_true_test, y_pred_test, average='macro', zero_division=0))
}

with open(os.path.join(OUT_DIR, "test_results.json"), "w") as f:
    json.dump(test_metrics, f, indent=2)

print("\n===== FINAL TEST EVALUATION RESULTS =====")
for metric, val in test_metrics.items():
    print(f"{metric.capitalize()}: {val:.4f}")

print(f"\nPipeline execution complete! All artifacts successfully saved to: {OUT_DIR}")